In [1]:
# -*- coding: utf-8 -*-
"""
NCAAB Prediction Model with Rolling Averages and Elo Ratings
"""

import os
import pandas as pd
import numpy as np
import math
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split # Using this for simplicity, but chronological split is better for time series

# --- Configuration ---
DATA_DIRECTORY = 'ncaa_box_scores_2024_25'
SCHEDULE_FILE = 'schedule.csv' # Needed for Elo calculation to find home team easily if not in box score reliably
SPLIT_DATE = '2025-02-28'
ROLLING_WINDOW_SIZE = 3 # Games to average over

# Elo Parameters
INITIAL_ELO = 1500
K_FACTOR = 20
HOME_ADVANTAGE_ELO = 100 # Elo points added to home team rating for prediction

# --- Step 1: Load Data ---
print("--- Step 1: Loading Data ---")
dataframes = []
if not os.path.exists(DATA_DIRECTORY):
    raise FileNotFoundError(f"Data directory not found: {DATA_DIRECTORY}")

for file_name in os.listdir(DATA_DIRECTORY):
    file_path = os.path.join(DATA_DIRECTORY, file_name)
    if os.path.isfile(file_path) and file_path.endswith('.csv'):
        try:
            df = pd.read_csv(file_path)
            # Basic check for essential columns (adjust if needed)
            if 'game_id' in df.columns and 'team' in df.columns and 'player' in df.columns:
                 dataframes.append(df)
            else:
                 print(f"Skipping {file_name}: Missing essential columns.")
        except Exception as e:
            print(f"Error reading {file_path}: {e}")

if not dataframes:
    raise ValueError(f"No valid CSV files found or read from the directory '{DATA_DIRECTORY}'.")

raw_dataframe = pd.concat(dataframes, ignore_index=True)
print(f"Loaded {len(raw_dataframe)} rows from {len(dataframes)} files.")

# --- Step 2: Basic Cleaning & Preprocessing ---
print("\n--- Step 2: Basic Cleaning & Preprocessing ---")
# Filter for TEAM rows
team_dataframe = raw_dataframe[raw_dataframe['player'] == 'TEAM'].copy()

# Convert date and sort
team_dataframe['date'] = pd.to_datetime(team_dataframe['date'])
team_dataframe = team_dataframe.sort_values(by=['date', 'game_id', 'team']) # Sort crucial for sequential processing

# Define result numeric (1 for Win, 0 for Loss)
team_dataframe['result_numeric'] = (team_dataframe['result'] == 'W').astype(int)

# Drop unnecessary columns (adjust if needed)
cols_to_drop = ['player_id', 'player', 'MIN', 'starter', 'location', 'result']
team_dataframe = team_dataframe.drop(columns=cols_to_drop, errors='ignore')

# Add Opponent's Actual Stats to each row (for calculating stats allowed)
opp_stats_temp = team_dataframe[[
    'game_id', 'team', 'FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA',
    'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS'
]].copy()
opp_stats_temp = opp_stats_temp.add_prefix('opp_')
opp_stats_temp.rename(columns={'opp_game_id': 'game_id', 'opp_team': 'opponent_team_actual'}, inplace=True)

# Merge opponent stats
team_dataframe = pd.merge(
    team_dataframe,
    opp_stats_temp,
    how='left',
    on='game_id',
    suffixes=('', '_y_drop') # Suffix for columns from the right side of merge
)
# Filter out self-merge rows and keep only the row where the opponent stats match the actual opponent
team_dataframe = team_dataframe[team_dataframe['opponent'] == team_dataframe['opponent_team_actual']].copy()
# Drop merge helper columns and any duplicated columns from merge
team_dataframe = team_dataframe.drop(columns=[col for col in team_dataframe.columns if '_y_drop' in col or col == 'opponent_team_actual'])

# Rename opponent stats columns for clarity (these are stats *allowed* by the 'team' in that row)
opp_allowed_cols_map = {col: col.replace('opp_', 'opp_allowed_') for col in opp_stats_temp.columns if col.startswith('opp_') and col != 'opp_game_id' and col != 'opp_team'}
team_dataframe.rename(columns=opp_allowed_cols_map, inplace=True)

print(f"Cleaned team data shape: {team_dataframe.shape}")
print("Columns after adding opponent allowed stats:", team_dataframe.columns)

# --- Step 3: Elo Rating Calculation ---
print("\n--- Step 3: Calculating Elo Ratings ---")

def calculate_expected_outcome(rating1, rating2):
    return 1 / (1 + 10**((rating2 - rating1) / 400))

# Ensure dataframe is sorted for chronological Elo updates
team_dataframe = team_dataframe.sort_values(by=['date', 'game_id', 'team']).reset_index(drop=True)

teams = pd.concat([team_dataframe['team'], team_dataframe['opponent']]).unique()
elo_ratings = {team: INITIAL_ELO for team in teams}
game_elos = {} # Store calculated Elos for each game to avoid recalculating

# Add columns to store PRE-GAME Elo
team_dataframe['team_elo_before'] = np.nan
team_dataframe['opp_elo_before'] = np.nan

# Iterate through unique games chronologically
unique_games = team_dataframe[['date', 'game_id']].drop_duplicates().sort_values(by=['date', 'game_id'])

for index, game_info in unique_games.iterrows():
    game_id = game_info['game_id']
    game_rows = team_dataframe[team_dataframe['game_id'] == game_id]

    if len(game_rows) != 2:
        # print(f"Skipping game_id {game_id}: Found {len(game_rows)} rows instead of 2.")
        continue # Skip games with incomplete data

    row1 = game_rows.iloc[0]
    row2 = game_rows.iloc[1]

    team1 = row1['team']
    team2 = row1['opponent'] # Which is row2['team']

    # Get current Elo ratings
    elo1 = elo_ratings.get(team1, INITIAL_ELO)
    elo2 = elo_ratings.get(team2, INITIAL_ELO)

    # Store PRE-GAME Elo ratings in the main DataFrame
    team_dataframe.loc[row1.name, 'team_elo_before'] = elo1
    team_dataframe.loc[row1.name, 'opp_elo_before'] = elo2
    team_dataframe.loc[row2.name, 'team_elo_before'] = elo2 # Team for row2 is team2
    team_dataframe.loc[row2.name, 'opp_elo_before'] = elo1 # Opponent for row2 is team1

    # Determine home team and apply advantage for prediction
    home_team = team1 if row1['home'] else team2
    away_team = team2 if row1['home'] else team1

    elo_home = elo_ratings.get(home_team, INITIAL_ELO) + HOME_ADVANTAGE_ELO
    elo_away = elo_ratings.get(away_team, INITIAL_ELO)

    # Calculate expected outcome (from home team's perspective)
    expected_home_win = calculate_expected_outcome(elo_home, elo_away)

    # Determine actual outcome (1 if home wins, 0 if away wins)
    # Use the numeric result column created earlier
    actual_home_win = row1['result_numeric'] if row1['home'] else row2['result_numeric']

    # Update Elo ratings
    update_factor_home = K_FACTOR * (actual_home_win - expected_home_win)
    elo_ratings[home_team] += update_factor_home
    elo_ratings[away_team] -= update_factor_home # Away update is opposite

# Drop rows where Elo wasn't calculated (e.g., first game, games with issues)
initial_rows = len(team_dataframe)
team_dataframe.dropna(subset=['team_elo_before', 'opp_elo_before'], inplace=True)
print(f"Dropped {initial_rows - len(team_dataframe)} rows with missing Elo ratings.")
print("Elo calculation complete.")
print(team_dataframe[['date', 'team', 'opponent', 'team_elo_before', 'opp_elo_before']].tail())

# --- Step 4: Calculate Rolling Averages (Using PREVIOUS Games) ---
print("\n--- Step 4: Calculating Rolling Averages ---")

stats_cols_base = ['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'team_score', 'opponent_score']
opp_allowed_cols = [col for col in team_dataframe.columns if col.startswith('opp_allowed_')]
all_stats_cols = stats_cols_base + opp_allowed_cols

rolling_feature_cols = []

for col in all_stats_cols:
    new_col_name = f'{col}_roll{ROLLING_WINDOW_SIZE}'
    team_dataframe[new_col_name] = team_dataframe.groupby('team')[col].transform(
        lambda x: x.shift(1).rolling(window=ROLLING_WINDOW_SIZE, min_periods=1).mean()
    )
    rolling_feature_cols.append(new_col_name)

# Calculate derived rolling averages (handle division by zero)
team_dataframe[f'FG%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'FGM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'FGA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'3PT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'3PTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'3PTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'FT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'FTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'FTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'opp_allowed_FGM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'opp_allowed_FGA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)
team_dataframe[f'opp_allowed_3PT%_roll{ROLLING_WINDOW_SIZE}'] = (team_dataframe[f'opp_allowed_3PTM_roll{ROLLING_WINDOW_SIZE}'] / team_dataframe[f'opp_allowed_3PTA_roll{ROLLING_WINDOW_SIZE}']).fillna(0)

rolling_feature_cols.extend([f'FG%_roll{ROLLING_WINDOW_SIZE}', f'3PT%_roll{ROLLING_WINDOW_SIZE}', f'FT%_roll{ROLLING_WINDOW_SIZE}',
                             f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}', f'opp_allowed_3PT%_roll{ROLLING_WINDOW_SIZE}'])

# Drop rows where rolling average couldn't be calculated (first N games)
initial_rows = len(team_dataframe)
team_dataframe.dropna(subset=[f'FGM_roll{ROLLING_WINDOW_SIZE}'], inplace=True) # Check one key column
print(f"Dropped {initial_rows - len(team_dataframe)} rows with missing rolling averages.")
print("Rolling averages calculated.")
print(team_dataframe[['date', 'team', 'opponent', f'PTS_roll{ROLLING_WINDOW_SIZE}', f'FG%_roll{ROLLING_WINDOW_SIZE}']].tail())


# --- Step 5: Create Matchup DataFrame ---
print("\n--- Step 5: Creating Matchup DataFrame ---")

# Columns needed for merging and final features/labels
cols_for_matchup = ['game_id', 'date', 'team', 'opponent', 'home', 'result_numeric',
                    'team_elo_before', 'opp_elo_before'] + rolling_feature_cols

matchup_data = team_dataframe[cols_for_matchup].copy()

# Self-merge to get opponent's stats on the same row
matchup_merged = pd.merge(
    matchup_data,
    matchup_data,
    on='game_id',
    suffixes=('_team', '_opp') # _team belongs to the primary team, _opp to their opponent
)

# Filter for valid matchups (team A vs team B, not A vs A)
# Ensure team names align correctly after merge
matchups_final = matchup_merged[matchup_merged['team_team'] != matchup_merged['team_opp']].copy()
matchups_final = matchups_final[matchups_final['team_team'] == matchups_final['opponent_opp']].copy()

# Create unique game identifier to drop duplicates (A vs B is same as B vs A)
matchups_final['game_pair_id'] = matchups_final.apply(
    lambda row: f"{row['game_id']}_{'_'.join(sorted([row['team_team'], row['team_opp']]))}", axis=1
)
matchups_final = matchups_final.drop_duplicates(subset='game_pair_id')
matchups_final.drop(columns=['game_pair_id'], inplace=True) # Drop helper column

# Define the label: Did team_team win?
matchups_final['label'] = matchups_final['result_numeric_team'] # Use the numeric result column

print(f"Created final matchup data with shape: {matchups_final.shape}")
print("Matchup columns:", matchups_final.columns)


# --- Step 6: Feature Engineering (Differences) ---
print("\n--- Step 6: Engineering Difference Features ---")

feature_cols_final = []

# Elo Difference
matchups_final['elo_diff'] = matchups_final['team_elo_before_team'] - matchups_final['team_elo_before_opp'] # Note: team_elo_before_opp is opp's Elo
feature_cols_final.append('elo_diff')

# Rolling Average Differences
stats_to_diff = ['PTS', 'team_score', 'opponent_score', 'FG%', '3PT%', 'FT%', 'REB', 'AST', 'TO', 'STL', 'BLK', 'PF', 'opp_allowed_FG%', 'opp_allowed_PTS']
for stat in stats_to_diff:
    col_team = f'{stat}_roll{ROLLING_WINDOW_SIZE}_team'
    col_opp = f'{stat}_roll{ROLLING_WINDOW_SIZE}_opp'
    if col_team in matchups_final.columns and col_opp in matchups_final.columns:
         diff_col_name = f'{stat}_roll_diff'
         matchups_final[diff_col_name] = matchups_final[col_team] - matchups_final[col_opp]
         feature_cols_final.append(diff_col_name)
    else:
        print(f"Warning: Columns for {stat} difference not found.")

# Offense vs Defense Features
matchups_final['team_off_vs_opp_def_fg%'] = matchups_final[f'FG%_roll{ROLLING_WINDOW_SIZE}_team'] - matchups_final[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}_opp']
feature_cols_final.append('team_off_vs_opp_def_fg%')

matchups_final['opp_off_vs_team_def_fg%'] = matchups_final[f'FG%_roll{ROLLING_WINDOW_SIZE}_opp'] - matchups_final[f'opp_allowed_FG%_roll{ROLLING_WINDOW_SIZE}_team']
feature_cols_final.append('opp_off_vs_team_def_fg%')

# Home Advantage Feature
matchups_final['team_is_home'] = matchups_final['home_team'].astype(int)
feature_cols_final.append('team_is_home')

# Final check for NaNs in selected features and fill
matchups_final[feature_cols_final] = matchups_final[feature_cols_final].fillna(0)

print(f"Engineered {len(feature_cols_final)} features.")
print("Final features:", feature_cols_final)
print(matchups_final[['date_team', 'team_team', 'team_opp', 'elo_diff', f'PTS_roll_diff', 'team_off_vs_opp_def_fg%', 'label']].head())


# --- Step 7: Train/Test Split ---
print("\n--- Step 7: Splitting Data ---")
split_timestamp = pd.Timestamp(SPLIT_DATE)

# Ensure date column is correct for splitting
date_col_to_use = 'date_team' if 'date_team' in matchups_final.columns else 'date'
if date_col_to_use not in matchups_final.columns:
    raise KeyError(f"Date column '{date_col_to_use}' not found for splitting.")

matchups_final[date_col_to_use] = pd.to_datetime(matchups_final[date_col_to_use])

train_df = matchups_final[matchups_final[date_col_to_use] < split_timestamp].copy()
test_df = matchups_final[matchups_final[date_col_to_use] >= split_timestamp].copy()

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

if train_df.empty or test_df.empty:
    raise ValueError("Train or test split resulted in empty DataFrame. Check SPLIT_DATE and data range.")

X_train = train_df[feature_cols_final]
y_train = train_df['label']
X_test = test_df[feature_cols_final]
y_test = test_df['label']

# Final check: Fill NaNs using training data mean/median if any slipped through
# (Should ideally be handled per feature type, but mean is often okay for RF)
if X_train.isnull().sum().sum() > 0:
    print("Warning: NaNs found in X_train before training. Filling with mean.")
    X_train = X_train.fillna(X_train.mean())
if X_test.isnull().sum().sum() > 0:
    print("Warning: NaNs found in X_test before evaluation. Filling with training mean.")
    X_test = X_test.fillna(X_train.mean()) # Use training mean


# --- Step 8: Model Training ---
print("\n--- Step 8: Training Model ---")
model = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10, min_samples_leaf=5, n_jobs=-1, class_weight='balanced')
model.fit(X_train, y_train)
print("Model training complete.")

# Feature Importance
importances = pd.DataFrame({'feature': X_train.columns, 'importance': model.feature_importances_})
importances = importances.sort_values('importance', ascending=False).reset_index(drop=True)
print("\nFeature Importances:")
print(importances.head(15)) # Show top 15


# --- Step 9: Evaluation ---
print("\n--- Step 9: Evaluating Model ---")
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1] # Probability of class 1 (team_team wins)

accuracy = accuracy_score(y_test, y_pred)
report = classification_report(y_test, y_pred)

print(f"\nTest Set Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(report)


# --- Step 10: Display Results ---
print("\n--- Step 10: Prediction Results Display ---")
results_df = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
results_df['predicted_winner'] = np.where(y_pred == 1, results_df['team_team'], results_df['team_opp'])
results_df['actual_winner'] = np.where(y_test == 1, results_df['team_team'], results_df['team_opp'])
# Confidence: Use probability of the predicted class
results_df['confidence'] = np.where(y_pred == 1, y_proba, 1 - y_proba)
results_df['correct'] = (y_pred == y_test)
results_df['status'] = results_df['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')


print("\n📋 Prediction Results with Confidence (Probability of Predicted Outcome):")
# Display key columns, sorted by date
display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
print(results_df[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).to_string(index=False))

print("\n--- Script Finished ---")

--- Step 1: Loading Data ---
Loaded 132003 rows from 364 files.

--- Step 2: Basic Cleaning & Preprocessing ---
Cleaned team data shape: (11356, 38)
Columns after adding opponent allowed stats: Index(['FGM', 'FGA', '3PTM', '3PTA', 'FTM', 'FTA', 'OREB', 'DREB', 'REB',
       'AST', 'STL', 'BLK', 'TO', 'PF', 'PTS', 'team', 'opponent', 'home',
       'game_id', 'date', 'team_score', 'opponent_score', 'result_numeric',
       'opp_allowed_FGM', 'opp_allowed_FGA', 'opp_allowed_3PTM',
       'opp_allowed_3PTA', 'opp_allowed_FTM', 'opp_allowed_FTA',
       'opp_allowed_OREB', 'opp_allowed_DREB', 'opp_allowed_REB',
       'opp_allowed_AST', 'opp_allowed_STL', 'opp_allowed_BLK',
       'opp_allowed_TO', 'opp_allowed_PF', 'opp_allowed_PTS'],
      dtype='object')

--- Step 3: Calculating Elo Ratings ---
Dropped 64 rows with missing Elo ratings.
Elo calculation complete.
            date       team  opponent  team_elo_before  opp_elo_before
11351 2025-04-05  Villanova       UCF      1556.358299  

In [2]:
# Cell: Model 1 - Random Forest (Baseline)
print("\n--- Model 1: Random Forest Classifier ---")
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd # Ensure pandas is imported if running cells independently

# --- Model Training ---
model_rf = RandomForestClassifier(n_estimators=200, random_state=42, max_depth=10, min_samples_leaf=5, n_jobs=-1, class_weight='balanced')
model_rf.fit(X_train, y_train)
print("Random Forest training complete.")

# --- Prediction ---
y_pred_rf = model_rf.predict(X_test)
y_proba_rf = model_rf.predict_proba(X_test) # Probabilities for both classes

# --- Evaluation ---
accuracy_rf = accuracy_score(y_test, y_pred_rf)
report_rf = classification_report(y_test, y_pred_rf)

print(f"\nRandom Forest Test Set Accuracy: {accuracy_rf:.4f}")
print("\nRandom Forest Classification Report:")
print(report_rf)

# --- Display Results ---
results_df_rf = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
results_df_rf['predicted_winner'] = np.where(y_pred_rf == 1, results_df_rf['team_team'], results_df_rf['team_opp'])
results_df_rf['actual_winner'] = np.where(y_test == 1, results_df_rf['team_team'], results_df_rf['team_opp'])
# Confidence: Use probability of the predicted class
results_df_rf['confidence'] = np.where(y_pred_rf == 1, y_proba_rf[:, 1], y_proba_rf[:, 0]) # Use prob of class 0 if predicting 0
results_df_rf['correct'] = (y_pred_rf == y_test)
results_df_rf['status'] = results_df_rf['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')

print("\n📋 RF Prediction Results with Confidence (Probability of Predicted Outcome):")
display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
print(results_df_rf[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).head(15).to_string(index=False)) # Show first 15
print("...")
print(results_df_rf[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).tail(15).to_string(index=False)) # Show last 15


--- Model 1: Random Forest Classifier ---
Random Forest training complete.

Random Forest Test Set Accuracy: 0.6928

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.68      0.70      0.69       389
           1       0.70      0.69      0.69       402

    accuracy                           0.69       791
   macro avg       0.69      0.69      0.69       791
weighted avg       0.69      0.69      0.69       791


📋 RF Prediction Results with Confidence (Probability of Predicted Outcome):
 date_team        team_team         team_opp predicted_winner    actual_winner  confidence    status
2025-02-28        Fairfield Mount St. Mary's Mount St. Mary's        Fairfield       0.736   ❌ Wrong
2025-02-28         Canisius        Manhattan        Manhattan        Manhattan       0.772 ✅ Correct
2025-02-28             Iona          Niagara             Iona             Iona       0.583 ✅ Correct
2025-02-28         Davidson          

In [3]:
# Cell: Model 2 - Logistic Regression (with Scaling)
print("\n--- Model 2: Logistic Regression ---")
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd # Ensure pandas is imported

# --- Feature Scaling ---
# Logistic Regression benefits from scaled features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train) # Fit ONLY on training data
X_test_scaled = scaler.transform(X_test)      # Transform test data
print("Features scaled for Logistic Regression.")

# --- Model Training ---
model_lr = LogisticRegression(random_state=42, class_weight='balanced', max_iter=1000) # Increased max_iter for convergence
model_lr.fit(X_train_scaled, y_train)
print("Logistic Regression training complete.")

# --- Prediction ---
y_pred_lr = model_lr.predict(X_test_scaled)
y_proba_lr = model_lr.predict_proba(X_test_scaled)

# --- Evaluation ---
accuracy_lr = accuracy_score(y_test, y_pred_lr)
report_lr = classification_report(y_test, y_pred_lr)

print(f"\nLogistic Regression Test Set Accuracy: {accuracy_lr:.4f}")
print("\nLogistic Regression Classification Report:")
print(report_lr)

# --- Display Results ---
results_df_lr = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
results_df_lr['predicted_winner'] = np.where(y_pred_lr == 1, results_df_lr['team_team'], results_df_lr['team_opp'])
results_df_lr['actual_winner'] = np.where(y_test == 1, results_df_lr['team_team'], results_df_lr['team_opp'])
results_df_lr['confidence'] = np.where(y_pred_lr == 1, y_proba_lr[:, 1], y_proba_lr[:, 0])
results_df_lr['correct'] = (y_pred_lr == y_test)
results_df_lr['status'] = results_df_lr['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')

print("\n📋 LR Prediction Results with Confidence (Probability of Predicted Outcome):")
display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
print(results_df_lr[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).head(15).to_string(index=False)) # Show first 15
print("...")
print(results_df_lr[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).tail(15).to_string(index=False)) # Show last 15


--- Model 2: Logistic Regression ---
Features scaled for Logistic Regression.
Logistic Regression training complete.

Logistic Regression Test Set Accuracy: 0.7143

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.73      0.71       389
           1       0.73      0.70      0.71       402

    accuracy                           0.71       791
   macro avg       0.71      0.71      0.71       791
weighted avg       0.71      0.71      0.71       791


📋 LR Prediction Results with Confidence (Probability of Predicted Outcome):
 date_team        team_team         team_opp predicted_winner    actual_winner  confidence    status
2025-02-28        Fairfield Mount St. Mary's Mount St. Mary's        Fairfield       0.654   ❌ Wrong
2025-02-28         Canisius        Manhattan        Manhattan        Manhattan       0.788 ✅ Correct
2025-02-28             Iona          Niagara             Iona             Iona      

In [4]:
# Cell: Model 3 - LightGBM Classifier
print("\n--- Model 3: LightGBM Classifier ---")
# Make sure lightgbm is installed: pip install lightgbm
try:
    import lightgbm as lgb
except ImportError:
    print("LightGBM not found. Please install it: pip install lightgbm")
    # Assign dummy values to prevent errors in the summary cell if LGBM fails
    accuracy_lgbm = 0.0
    report_lgbm = "LightGBM not installed."
    results_df_lgbm = pd.DataFrame() # Empty dataframe
else:
    from sklearn.metrics import accuracy_score, classification_report
    import numpy as np
    import pandas as pd # Ensure pandas is imported

    # --- Model Training ---
    # LightGBM doesn't strictly require scaling
    model_lgbm = lgb.LGBMClassifier(random_state=42, class_weight='balanced', n_jobs=-1)
    model_lgbm.fit(X_train, y_train)
    print("LightGBM training complete.")

    # --- Prediction ---
    y_pred_lgbm = model_lgbm.predict(X_test)
    y_proba_lgbm = model_lgbm.predict_proba(X_test)

    # --- Evaluation ---
    accuracy_lgbm = accuracy_score(y_test, y_pred_lgbm)
    report_lgbm = classification_report(y_test, y_pred_lgbm)

    print(f"\nLightGBM Test Set Accuracy: {accuracy_lgbm:.4f}")
    print("\nLightGBM Classification Report:")
    print(report_lgbm)

    # --- Display Results ---
    results_df_lgbm = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
    results_df_lgbm['predicted_winner'] = np.where(y_pred_lgbm == 1, results_df_lgbm['team_team'], results_df_lgbm['team_opp'])
    results_df_lgbm['actual_winner'] = np.where(y_test == 1, results_df_lgbm['team_team'], results_df_lgbm['team_opp'])
    results_df_lgbm['confidence'] = np.where(y_pred_lgbm == 1, y_proba_lgbm[:, 1], y_proba_lgbm[:, 0])
    results_df_lgbm['correct'] = (y_pred_lgbm == y_test)
    results_df_lgbm['status'] = results_df_lgbm['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')

    print("\n📋 LGBM Prediction Results with Confidence (Probability of Predicted Outcome):")
    display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
    print(results_df_lgbm[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).head(15).to_string(index=False)) # Show first 15
    print("...")
    print(results_df_lgbm[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).tail(15).to_string(index=False)) # Show last 15


--- Model 3: LightGBM Classifier ---
[LightGBM] [Info] Number of positive: 2317, number of negative: 2317
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000338 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3696
[LightGBM] [Info] Number of data points in the train set: 4634, number of used features: 18
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
LightGBM training complete.

LightGBM Test Set Accuracy: 0.6941

LightGBM Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.70      0.69       389
           1       0.70      0.69      0.70       402

    accuracy                           0.69       791
   macro avg       0.69      0.69      0.69       791
weighted avg       0.69      0.69      0.69       791


📋 LGBM Prediction Results with Confid

In [5]:
# Cell: Model 4 - XGBoost Classifier
print("\n--- Model 4: XGBoost Classifier ---")
# Make sure xgboost is installed: pip install xgboost
try:
    import xgboost as xgb
except ImportError:
    print("XGBoost not found. Please install it: pip install xgboost")
    # Assign dummy values to prevent errors in the summary cell if XGBoost fails
    accuracy_xgb = 0.0
    report_xgb = "XGBoost not installed."
    results_df_xgb = pd.DataFrame() # Empty dataframe
else:
    from sklearn.metrics import accuracy_score, classification_report
    import numpy as np
    import pandas as pd # Ensure pandas is imported

    # --- Model Training ---
    # XGBoost often benefits from label encoding the target if it's not 0/1, but ours is.
    # It can handle NaNs internally, but we've already filled them.
    # Use scale_pos_weight for imbalanced classes if needed, similar to class_weight='balanced'
    # Calculate scale_pos_weight: sum(negative instances) / sum(positive instances)
    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model_xgb = xgb.XGBClassifier(
        objective='binary:logistic', # for binary classification
        eval_metric='logloss',       # evaluation metric
        use_label_encoder=False,     # recommended setting
        random_state=42,
        n_estimators=100,            # Number of boosting rounds (can tune)
        learning_rate=0.1,           # Step size shrinkage (can tune)
        max_depth=3,                 # Max depth of a tree (can tune)
        subsample=0.8,               # Fraction of samples used per tree
        colsample_bytree=0.8,        # Fraction of features used per tree
        scale_pos_weight=scale_pos_weight, # Handle class imbalance
        n_jobs=-1                    # Use all available CPU cores
    )

    model_xgb.fit(X_train, y_train)
    print("XGBoost training complete.")

    # --- Prediction ---
    y_pred_xgb = model_xgb.predict(X_test)
    y_proba_xgb = model_xgb.predict_proba(X_test)

    # --- Evaluation ---
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    report_xgb = classification_report(y_test, y_pred_xgb)

    print(f"\nXGBoost Test Set Accuracy: {accuracy_xgb:.4f}")
    print("\nXGBoost Classification Report:")
    print(report_xgb)

    # --- Display Results ---
    results_df_xgb = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
    results_df_xgb['predicted_winner'] = np.where(y_pred_xgb == 1, results_df_xgb['team_team'], results_df_xgb['team_opp'])
    results_df_xgb['actual_winner'] = np.where(y_test == 1, results_df_xgb['team_team'], results_df_xgb['team_opp'])
    results_df_xgb['confidence'] = np.where(y_pred_xgb == 1, y_proba_xgb[:, 1], y_proba_xgb[:, 0])
    results_df_xgb['correct'] = (y_pred_xgb == y_test)
    results_df_xgb['status'] = results_df_xgb['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')

    print("\n📋 XGBoost Prediction Results with Confidence (Probability of Predicted Outcome):")
    display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
    print(results_df_xgb[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).head(15).to_string(index=False)) # Show first 15
    print("...")
    print(results_df_xgb[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).tail(15).to_string(index=False)) # Show last 15


--- Model 4: XGBoost Classifier ---
XGBoost training complete.

XGBoost Test Set Accuracy: 0.7042

XGBoost Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.71      0.70       389
           1       0.72      0.69      0.70       402

    accuracy                           0.70       791
   macro avg       0.70      0.70      0.70       791
weighted avg       0.70      0.70      0.70       791


📋 XGBoost Prediction Results with Confidence (Probability of Predicted Outcome):
 date_team        team_team         team_opp predicted_winner    actual_winner  confidence    status
2025-02-28        Fairfield Mount St. Mary's Mount St. Mary's        Fairfield       0.771   ❌ Wrong
2025-02-28         Canisius        Manhattan        Manhattan        Manhattan       0.794 ✅ Correct
2025-02-28             Iona          Niagara             Iona             Iona       0.511 ✅ Correct
2025-02-28         Davidson              VCU            

/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [20:54:23] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [6]:
# Cell: Summary of Model Performance (Updated)
print("\n--- Model Performance Summary ---")
print(f"Random Forest Accuracy:     {accuracy_rf:.4f}")
print(f"Logistic Regression Accuracy: {accuracy_lr:.4f}")
if 'accuracy_lgbm' in locals():
    print(f"LightGBM Accuracy:          {accuracy_lgbm:.4f}")
else:
    print("LightGBM Accuracy:          N/A (not run)")
if 'accuracy_xgb' in locals(): # Check if XGBoost ran successfully
    print(f"XGBoost Accuracy:           {accuracy_xgb:.4f}")
else:
    print("XGBoost Accuracy:           N/A (not run/installed)")


print("\n--- Random Forest Report ---")
print(report_rf)
print("\n--- Logistic Regression Report ---")
print(report_lr)
if 'report_lgbm' in locals():
    print("\n--- LightGBM Report ---")
    print(report_lgbm)
if 'report_xgb' in locals():
    print("\n--- XGBoost Report ---")
    print(report_xgb)


--- Model Performance Summary ---
Random Forest Accuracy:     0.6928
Logistic Regression Accuracy: 0.7143
LightGBM Accuracy:          0.6941
XGBoost Accuracy:           0.7042

--- Random Forest Report ---
              precision    recall  f1-score   support

           0       0.68      0.70      0.69       389
           1       0.70      0.69      0.69       402

    accuracy                           0.69       791
   macro avg       0.69      0.69      0.69       791
weighted avg       0.69      0.69      0.69       791


--- Logistic Regression Report ---
              precision    recall  f1-score   support

           0       0.70      0.73      0.71       389
           1       0.73      0.70      0.71       402

    accuracy                           0.71       791
   macro avg       0.71      0.71      0.71       791
weighted avg       0.71      0.71      0.71       791


--- LightGBM Report ---
              precision    recall  f1-score   support

           0       0.69

In [7]:
# Cell: Model - Neural Network (TensorFlow/Keras)
print("\n--- Model: Neural Network (TensorFlow/Keras) ---")
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except ImportError:
    print("TensorFlow not found. Please install it: pip install tensorflow")
    # Assign dummy values if TensorFlow is not installed
    accuracy_nn = 0.0
    report_nn = "TensorFlow not installed."
    results_df_nn = pd.DataFrame()
    models['NN'] = None # Indicate model wasn't trained
else:
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import accuracy_score, classification_report
    import numpy as np
    import pandas as pd # Ensure pandas is imported

    # --- Feature Scaling (Important for NNs) ---
    scaler_nn = StandardScaler()
    X_train_scaled = scaler_nn.fit_transform(X_train) # Fit ONLY on training data
    X_test_scaled = scaler_nn.transform(X_test)      # Transform test data
    print("Features scaled for Neural Network.")
    print(f"Scaled training data shape: {X_train_scaled.shape}")

    # --- Build the Neural Network Model ---
    def build_nn_model(input_shape):
        model = keras.Sequential(
            [
                layers.Input(shape=(input_shape,)), # Define input shape explicitly
                layers.Dense(64, activation="relu", name="layer1"),
                layers.Dropout(0.3), # Regularization to prevent overfitting
                layers.Dense(32, activation="relu", name="layer2"),
                layers.Dropout(0.2),
                layers.Dense(1, activation="sigmoid", name="output_layer") # Sigmoid for binary probability
            ]
        )
        return model

    nn_model = build_nn_model(X_train_scaled.shape[1]) # Pass the number of features
    nn_model.summary() # Print model architecture

    # --- Compile the Model ---
    nn_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001), # Adam optimizer
        loss="binary_crossentropy",          # Loss function for binary classification
        metrics=["accuracy"]                 # Metric to monitor
    )

    # --- Prepare Callbacks (Optional but Recommended) ---
    # Stop training if validation loss doesn't improve for 'patience' epochs
    early_stopping = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10, # Number of epochs with no improvement after which training will be stopped.
        restore_best_weights=True # Restores model weights from the epoch with the best value of the monitored quantity.
    )

    # --- Train the Model ---
    print("\n--- Training Neural Network ---")
    # Use a portion of training data for validation during training
    history = nn_model.fit(
        X_train_scaled,
        y_train,
        epochs=100,           # Max number of epochs
        batch_size=32,        # Samples per gradient update
        validation_split=0.2, # Use 20% of training data for validation
        callbacks=[early_stopping], # Apply early stopping
        verbose=1             # Set to 1 to see progress per epoch, 0 for silent
    )
    print("Neural Network training complete.")

    # --- Evaluate the Model ---
    print("\n--- Evaluating Neural Network on Test Set ---")
    loss_nn, accuracy_nn = nn_model.evaluate(X_test_scaled, y_test, verbose=0)
    print(f"\nNeural Network Test Set Accuracy: {accuracy_nn:.4f}")
    print(f"Neural Network Test Set Loss: {loss_nn:.4f}")

    # --- Prediction ---
    y_proba_nn_raw = nn_model.predict(X_test_scaled) # Gives probabilities (sigmoid output)
    y_pred_nn = (y_proba_nn_raw > 0.5).astype("int32").flatten() # Convert probabilities to 0 or 1

    # --- Classification Report ---
    report_nn = classification_report(y_test, y_pred_nn, zero_division=0)
    print("\nNeural Network Classification Report:")
    print(report_nn)

    # --- Store results for summary ---
    accuracies['NN'] = accuracy_nn
    reports['NN'] = report_nn
    # Store probabilities of the *predicted* class for confidence
    confidence_nn = np.where(y_pred_nn == 1, y_proba_nn_raw.flatten(), 1 - y_proba_nn_raw.flatten())
    predictions['NN'] = {'pred': y_pred_nn, 'proba_raw': y_proba_nn_raw, 'confidence': confidence_nn}
    models['NN'] = nn_model # Store the trained model object

    # --- Display Results ---
    results_df_nn = test_df[[date_col_to_use, 'team_team', 'team_opp']].copy()
    results_df_nn['predicted_winner'] = np.where(y_pred_nn == 1, results_df_nn['team_team'], results_df_nn['team_opp'])
    results_df_nn['actual_winner'] = np.where(y_test == 1, results_df_nn['team_team'], results_df_nn['team_opp'])
    results_df_nn['confidence'] = confidence_nn
    results_df_nn['correct'] = (y_pred_nn == y_test)
    results_df_nn['status'] = results_df_nn['correct'].apply(lambda x: '✅ Correct' if x else '❌ Wrong')

    print("\n📋 NN Prediction Results with Confidence (Probability of Predicted Outcome):")
    display_cols = [date_col_to_use, 'team_team', 'team_opp', 'predicted_winner', 'actual_winner', 'confidence', 'status']
    # Use display options to prevent truncation
    with pd.option_context('display.max_rows', 20, 'display.max_columns', None, 'display.width', 1000): # Show more rows
        print(results_df_nn[display_cols].sort_values(date_col_to_use).round({'confidence': 3}).to_string(index=False))

# Ensure the script doesn't crash if TF wasn't imported
if 'accuracy_nn' not in locals(): accuracy_nn = 0.0
if 'report_nn' not in locals(): report_nn = "TensorFlow not installed."
if 'models' not in locals(): models = {} # Initialize if first model cell
if 'NN' not in models: models['NN'] = None


--- Model: Neural Network (TensorFlow/Keras) ---
Features scaled for Neural Network.
Scaled training data shape: (4634, 18)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 64)             │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output_layer (Dense)            │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,329 (13.00 KB)

 Non-trainable params: 0 (0.00 B)


--- Training Neural Network ---
Epoch 1/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.6121 - loss: 0.6518 - val_accuracy: 0.6667 - val_loss: 0.5955
Epoch 2/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - accuracy: 0.6813 - loss: 0.5967 - val_accuracy: 0.6753 - val_loss: 0.5836
Epoch 3/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 941us/step - accuracy: 0.6845 - loss: 0.5955 - val_accuracy: 0.6677 - val_loss: 0.5847
Epoch 4/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 804us/step - accuracy: 0.6851 - loss: 0.5798 - val_accuracy: 0.6731 - val_loss: 0.5799
Epoch 5/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 772us/step - accuracy: 0.6961 - loss: 0.5727 - val_accuracy: 0.6785 - val_loss: 0.5762
Epoch 6/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 796us/step - accuracy: 0.7049 - loss: 0.5697 - val_accuracy: 0.6742 - val_loss: 0.5826
Epoch 7/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 810us/step - accuracy: 0.7012 - loss: 0.5761 - val_accuracy: 0.6796 - val_loss: 0.5749
Epoch 8/100
116/116 ━━━━━━━━━━━━━━━━━━━━ 0s 808us/st

NameError: name 'accuracies' is not defined